# Day S2 — 돌연변이, 보존, 상동성: 서열이 달라지는 방식

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/supplement/day-S2.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/supplement/day-S2.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 점 돌연변이 분류기

In [ ]:
bases = "TCAG"
amino = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
codon_table = {b1 + b2 + b3: amino[16 * i + 4 * j + k]
               for i, b1 in enumerate(bases)
               for j, b2 in enumerate(bases)
               for k, b3 in enumerate(bases)}

PURINES = {"A", "G"}

def classify(ref_codon, alt_codon):
    """코돈 하나에서 일어난 점 돌연변이를 두 가지 기준으로 분류한다."""
    diffs = [(r, a) for r, a in zip(ref_codon, alt_codon) if r != a]
    assert len(diffs) == 1, "염기가 정확히 하나만 달라야 한다"
    ref_base, alt_base = diffs[0]
    same_family = (ref_base in PURINES) == (alt_base in PURINES)
    base_change = "transition" if same_family else "transversion"

    ref_aa, alt_aa = codon_table[ref_codon], codon_table[alt_codon]
    if ref_aa == alt_aa:
        effect = "synonymous"
    elif alt_aa == "*":
        effect = "nonsense"
    else:
        effect = "missense"
    return f"{ref_codon}→{alt_codon}  {ref_base}>{alt_base} {base_change:12s} {ref_aa}→{alt_aa} {effect}"

examples = [
    ("SNCA 코돈 53 (A53T)", "GCA", "ACA"),
    ("SNCA 코돈 30 (A30P)", "GCA", "CCA"),
    ("SNCA 코돈 46 (E46K)", "GAG", "AAG"),
    ("가상: 셋째 자리 변화", "GCA", "GCG"),
    ("가상: 종결 코돈 생성", "TAC", "TAA"),
]
for name, ref, alt in examples:
    print(f"{name:22s} {classify(ref, alt)}")

### 예시 2 — frameshift: 한 글자와 세 글자의 차이

In [ ]:
bases = "TCAG"
amino = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
codon_table = {b1 + b2 + b3: amino[16 * i + 4 * j + k]
               for i, b1 in enumerate(bases)
               for j, b2 in enumerate(bases)
               for k, b3 in enumerate(bases)}

def translate(dna):
    protein = ""
    for pos in range(0, len(dna) - 2, 3):
        aa = codon_table[dna[pos:pos + 3]]
        protein += aa
        if aa == "*":          # 종결 코돈을 만나면 멈춘다
            break
    return protein

original = "ATGGCTGAAAAAGGTGTTCTGGCAGCATAA"     # 가상 CDS (10코돈)
minus_one = original[:7] + original[8:]          # 8번째 염기 1개 결실
minus_three = original[:6] + original[9:]        # 셋째 코돈 3개 결실

print("원래     :", translate(original))
print("1염기 결실:", translate(minus_one))
print("3염기 결실:", translate(minus_three))

### 예시 3 — identity와 similarity 계산

In [ ]:
seq1 = "MKTEGVLAAQEKTK-QGVAEA"
seq2 = "MKSEGVVAAAEKTKEQGLQEA"

groups = ["ILVM", "FWY", "KRH", "DE", "STNQ", "AG"]   # 성질이 비슷한 아미노산 묶음(설명용)

def similar(a, b):
    return any(a in g and b in g for g in groups)

assert len(seq1) == len(seq2)
identical = 0
similar_count = 0
marks = ""
for a, b in zip(seq1, seq2):
    if a == "-" or b == "-":
        marks += " "
    elif a == b:
        identical += 1
        marks += "|"
    elif similar(a, b):
        similar_count += 1
        marks += ":"
    else:
        marks += "."

length = len(seq1)                       # gap도 정렬 길이에 포함한다
print(seq1)
print(marks)
print(seq2)
print(f"정렬 길이 {length}, 동일 {identical}, 유사(동일 제외) {similar_count}")
print(f"identity   = {identical / length:.1%}")
print(f"similarity = {(identical + similar_count) / length:.1%}")

## 직접 해보기

3. **셋째 자리 실험.** 아래 코드로 코돈 `GCA`의 각 자리를 다른 세 염기로 바꿨을 때 synonymous가 몇 번 나오는지 셉니다. 자리마다 결과가 어떻게 다른지 한 줄로 적습니다.

In [ ]:
ref = "GCA"
for pos in range(3):
    for base in "ACGT":
        if base != ref[pos]:
            alt = ref[:pos] + base + ref[pos + 1:]
            print(pos + 1, classify(ref, alt))